# Multiclass classification

When solving a classification problem, many existing machine learning models allow only two classes to be separated, usually referred to as "positive" and "negative". Positive patterns are commonly those related to what is to be detected, such as disease, alarm, or a type of object in an image. Negative patterns are often characterised by the absence of this characteristic that positive patterns have. To develop an ANN that classifies into two classes, a single neuron is needed in the output layer, with a logarithmic (or similar) sigmoidal transfer function, such that the output of the ANN will be between 0 and 1, and can be interpreted as the ANN's certainty in classifying a pattern as "positive". The classification into "negative" or "positive" is done in a simple way, by applying a threshold which is typically 0.5, although this can be changed.

However, there are many occasions when a system that is able to classify into more than two classes is desired. A simple example is a system that wants to classify an image according to whether a dog, cat or mouse, or some other type of animal is observed. In this case, you want to develop a 4-class classification system: "dog"/"cat"/"mouse"/"other". If an ANN to distinguish between these 3 animals is required, an output neuron for each class is needed, including the "other" class (4 output neurons in total).

In the multiclass classification scheme, as has been done in previous assignments, an encoding called one-hot-encoding is generally used, which is based on obtaining a boolean value for each pattern and each class, in such a way that each boolean value will be equal to 1 if that pattern belongs to that class, and 0 otherwise. When training an ANN with this scheme, each output neuron can be understood as a model specialised in classifying in a given class. In this type of networks, a linear transfer function is usually used in the output layer, whereby negative outputs indicate that a neuron does not classify the pattern into that class (i.e. from the point of view of that class it classifies it as "negative"), and positive outputs indicate that a neuron classifies the pattern as that class (i.e. from the point of view of that class it classifies it as "positive"). The absolute value of a neuron's output indicates that neuron's confidence in the classification. Finally, the softmax function receives these classification values and transforms them in such a way that they are between 0 and 1, and add up to 1, interpreted as the probability of belonging to each class. The pattern will be classified into the class whose output value is the highest. The softmax function is defined as follows: 

$$
softmax(y^i) = \frac{e^{y^i}}{\sum_j{e^{y^j}}}
$$

where $y^i$ is the output of the $i$-th neuron. For example, in a 3-class classification problem, if the outputs from the 3 neurons are `[2, 1, 0.2]`, they would classify the inputs as belonging to their respective classes, although the first one with much greater certainty. After applying the softmax function, the respective probabilities will be `[0.65, 0.24, 0.11]`, so the pattern will be classified as the first class.

In this way, the softmax function converts the real values produced by the output neurons into probability values, so that the more negative a value is (the more certainty of not belonging to that class), the closer it is to 0, and the more positive a value is (the more certainty of belonging to that class), the closer it is to 1. As indicated above, the sum of output probabilities will be equal to 1. Because of this fact, a fourth special class "other" is needed in the example above and in any other example where a pattern may not belong to any of the predefined classes.

### Question 4.2

> ❓ Why is this extra class necessary when using the softmax function?

`The extra “other” class is necessary because the softmax function forces all output probabilities to sum to 1.
If a pattern doesn’t clearly belong to any of the known classes, softmax still distributes the total probability among them, wrongly implying confidence in one.
Including an “other” class allows the network to assign probability mass to patterns outside the predefined categories, representing “none of the above” cases.
This makes the classification more realistic and avoids forcing every input into one of the known classes.` 

Tip: write in Julia `softmax([-1, -1, -0.2])`, and interpret the inputs (what does the vector `[-1, -1, -0.2]` represent and how is it interpreted?) and outputs of the function (how much do the values add up to? what does each say?). To use this function, import it from the Flux library.

In [20]:
using Flux

softmax([-1,-1,-0.2])

3-element Vector{Float64}:
 0.23665609135556676
 0.23665609135556676
 0.5266878172888664

`The vector [-1, -1, -0.2] represents the raw outputs (logits) from three output neurons of a neural network, each corresponding to a class. The higher the value, the more confident the network is that the input belongs to that class. When applying the softmax function from Flux, these values are converted into probabilities that sum to 1. The result is approximately [0.24, 0.24, 0.53], meaning the network assigns around 24% probability to the first and second classes, and 53% probability to the third class, which would be chosen as the predicted class.`

### Question 4.3

> ❓ Might it not be necessary to create the additional class? What modification would have to be made to the ANN? How would the output be interpreted? How would the output class be generated based on the outputs of the output neurons?

`It might not be necessary to create the additional class if the network is designed to handle uncertainty differently.
In that case, the ANN would need a mechanism to recognize when none of the classes have high confidence, such as setting a probability threshold.
If all outputs are below that threshold, the pattern could be classified as “unknown.”
The outputs would still represent probabilities for each class, and the predicted class would be the one with the highest probability, as long as it exceeds the chosen threshold.`

### Question 4.4

> ❓ In general, how does the output of a model have to be in order not to need this fourth class?

`In general, the model’s output should allow low probabilities for all classes when the input does not clearly belong to any of them.
This means the network must be able to express uncertainty without being forced to assign all probability mass to the known classes.
Such behavior can be achieved by using independent sigmoid outputs instead of softmax, so each class is evaluated separately and their probabilities don’t have to sum to 1.
In this case, no extra “other” class is needed.`

### Question 4.5

> ❓ Does a kNN model need this fourth class?

`No, a k-Nearest Neighbors (kNN) model does not need this fourth “other” class.
kNN assigns a class based on the majority label of the k nearest training samples, so it always picks one of the existing classes.
If the input is very different from all known examples, kNN may still force a choice, but this does not require an explicit “other” class—only a distance threshold could be added to reject uncertain classifications.`

### Question 4.6

> ❓ How many classes would be necessary if an ANN wanted to recognise those 3 types of animals, and, if it is not one of them, to say whether it is an animal or not? What if the model is a kNN?

`If an ANN is used, it would need four classes: one for each animal and one additional class for “other animals.”
To also detect when the input is not an animal, a fifth class (“not an animal”) would be needed.
So, in total, the ANN would have five output neurons.
If the model is a kNN, it would not need to explicitly define an extra class in advance.
It would classify based on the labels of the nearest samples; however, to identify “not an animal,” you’d need training examples labeled as such.
Without them, kNN cannot infer a new “not an animal” category on its own.`

Therefore, the "positive"/"negative" scheme no longer applies if more than two classes are required. The problem in these cases is that many of the machine learning models are only capable of separating two classes, so theoretically they could not be used. An example of such systems are Support Vector Machines (SVM), which are discussed in more detail in the theory class. Modifications have been made to the formulation of this model to allow multi-class classifications; however, in practice they are not commonly used, and instead a strategy that allows binary SVMs to be used to classify into multiple classes is often employed.

There are two main strategies for converting multi-class problems into binary classification problems. These strategies are called "one-against-one" or "one-against-all". Both are explained in theory class, but since "one-against-all" is much more widely used, this strategy will be used in the following.

The "one-against-all" strategy is based on generating L binary classifiers for a classification problem of L classes, one per class. In the l-th problem, class l must be separated from the rest, i.e., the patterns belonging to that class will be considered "positive", and those not belonging to it will be considered "negative". Continuing with the previous example of animals, 3 different classification problems would have to be solved: one to classify "dog"/"not dog", one to classify "cat"/"not cat", and one to classify "mouse"/"not mouse". Three classifiers would therefore be trained with the same inputs but with different desired outputs for each problem.

### Question 4.7

> ❓ In the previously described problem, 4 classes were used for these 3 animals, including the class "other". Why not train a classifier for this class in a "one-against-all" scheme?

`In the “one-against-all” scheme, each classifier learns to distinguish one specific class from all the others.
The “other” class, however, is not a single, well-defined category — it includes everything that is not dog, cat, or mouse, which can be extremely diverse.
Training a classifier for such a broad and heterogeneous group would be meaningless and unreliable, since it lacks consistent features.
Instead, each of the specific animal classifiers implicitly defines the “other” class as the negative examples in their training.`

Once the binary classifiers are trained, any given pattern is fed into all the classifiers and, depending on the output, a decision is made. If only one of the systems has positive output, or none of the three classifies it as positive, the decision is clear. However, sometimes more than one classifier will give a positive output for the same pattern. Fortunately, many classifiers give information about the level of certainty or confidence they have that the pattern is classified as "positive". If more than one binary model classifies the pattern as positive, it will be assigned to the class corresponding to the classifier that has a higher certainty in its classification.

### Question 4.8

> ❓ Would it be possible to use the outputs of those 3 classifiers as the input of the softmax function? What would be the consequences?

`Yes, it is possible to use the outputs of the three classifiers as input to a softmax function, but this has important consequences. Softmax would convert the outputs into probabilities that sum to 1, even though the original classifiers were independent and not meant to be combined this way. This can distort the confidence levels, especially if more than one classifier strongly signals positive, because softmax forces a “winner-takes-all” distribution. As a result, the absolute certainty from each classifier is lost or misrepresented. It is usually better to directly compare the raw confidence scores of the classifiers instead of applying softmax.`

### Question 4.9

> ❓ In general, when there are L classes and a pattern may not belong to any of them, what is the impact of using the softmax function on the outputs? In which cases could it be used? Why?

`When using the softmax function with L classes, the outputs are always normalized to sum to 1, so the model is forced to assign the input to one of the predefined classes, even if it does not truly belong to any of them. This means the network cannot naturally express “none of the above” unless an extra class is explicitly added for it.
Softmax can be used safely when it is guaranteed that every input belongs to one of the known classes, because then the normalized probabilities accurately reflect the relative confidence in each class. It should not be used when inputs may fall outside the predefined classes, unless a special “other” class is included to capture such cases.`

### Question 4.10

> ❓ The softmax function is useful to get a loss value to train the ANN. However, if it were not used in the animal example above, would the fourth class "other" be necessary?

`Yes, the fourth class “other” would still be necessary. Without softmax, the network’s outputs could still assign high values to the known classes even for inputs that don’t belong to them. The “other” class provides a way for the network to explicitly represent patterns that do not match any of the predefined animals, allowing correct classification of inputs outside the main categories.`

Finally, it is necessary to consider a different scenario when assigning patterns to classes. So far, and in most situations, the classes considered are mutually exclusive, i.e. in the example above, an animal is either a dog, a cat, a mouse, or none of the 3, but it cannot be of several classes at the same time. This is the most common case, but occasionally a problem will have classes that are not mutually exclusive. For example, when classifying animal sounds according to the animal that makes them, it may happen that several animals are mixed in one sound. In these cases, the use of a linear transfer function in the last layer together with the softmax function would not work, since, naturally, the sum of the probabilities of belonging to the classes may be greater than 1 (it may belong to several classes at the same time). For these cases, the scheme that can be used to train ANNs is to use logarithmic sigmoidal transfer functions in the last layer (instead of linear), which give an output between 0 and 1, and not to perform transformation using the softmax function. In this way, the final output of each output neuron is independent of the rest of the output neurons, and more than one can take values close to 1. The output of each neuron would again be interpreted as the probability of belonging to that class, but in this case the sum of the probabilities does not have to be 1 (they are independent). Not applying the softmax function has two advantages: the first, already mentioned, is that it allows classification into non-mutually exclusive classes; the second is that an additional class ("other" in the example above) is no longer needed for cases where a set of inputs may not belong to any of the given classes.

### Question 4.11

> ❓ Why is this extra class no longer needed?

`The extra “other” class is no longer needed because each output neuron is independent and can indicate whether the input belongs to its class without affecting the others. If an input does not belong to any class, all neurons can output values close to 0, naturally representing “none of the above.” There’s no need to explicitly define a separate “other” class, since the network can express the absence of all known classes directly through the independent sigmoidal outputs.`

Given a set of inputs, as always, it is classified into the class whose output neuron has shown the highest confidence. This scheme of non-mutually exclusive outputs is similar to the "one-against-all" scheme, in which one classifier per class is trained in parallel. The classifiers are independent and the final class is that of the classifier that has the highest certainty of belonging to that class. If all classifiers return "negative" as a classification and there is no possibility of not belonging to any class, the classifier with the lowest certainty of being negative is classified in the corresponding class. If all classifiers return "negative" as a classification and there is a possibility of non-class membership, it is simply classified as "other".

The following table shows a summary of the different scenarios when using an ANN to solve a classification problem. Note that in the case of binary classification, the possibility that a set of entries do not belong to any class is not considered, since in this case we would be in multi-class classification.



|                        |                       | Is extra<br>class added<br>if it can<br>not belong<br>to any of<br>the given? | Transfer<br>function<br>of the<br>output | Output<br>range<br>of the<br>neuron | Function<br>applied<br>to the<br>outputs<br>(additional layer) | Final<br>output<br>range | Which classes<br>it classifies<br>into? |
|------------------------|-----------------------|------------------------------------------|-----------------------------------------|-----------------------------------|----------------------------------------------------|------------------------|------------------------------------|
| **Binary<br>classification** |                       | –                                        | Logarithmic<br>sigmoid                  | [0,1]                             | –                                                  | [0,1]                 | "negative"/"positive" if the output<br>is greater or equal to a threshold<br>(0.5 for outputs in [0,1]) |
| **Multiclass**         | Mutually<br>exclusive<br>classes | Yes                                      | Linear                                  | [-inf, +inf]                      | softmax                                            | [0,1]                 | Class whose output is<br>closer to 1                 |
|                        | Not<br>mutually<br>exclusive<br>classes | No                                       | Logarithmic<br>sigmoid                  | [0,1]                             | –                                                  | [0,1]                 | Classes whose output<br>is greater or equal<br>to a threshold (0.5 for outputs<br>in [0,1]); can be several or none |


In the case of using a "one-against-all" strategy, this would be similar to the last row, except that the interval would not necessarily be `[0, 1]`, but would be conditioned by the model used, and therefore the threshold as well. For example, the outputs of a SVM range from $-\infty$ to $+\infty$, so the typical threshold is set to 0.

Another factor to consider when dealing with multiclass problems is the performance metric. Most of the metrics studied (PPV, sensitivity, etc.) correspond to binary classification problems. When the number of classes is greater than 2, these metrics can still be used; however, their use is slightly different.

When the number of classes is greater than two, the PPV, NPV, sensitivity and specificity metrics can be calculated separately for each class. Thus, from the point of view of a particular class, that class will be referred to as the positive class and the rest of classes will be put together in the negative class. In this way, from the exclusive point of view of that class, TP, TN, FP and FN can be calculated, and from them the sensitivity, specificity, PPV and NPV values for that particular class, and finally the F-score value. This way of treating classes separately is similar to the development of several classifiers in the "one-against-all" strategy (in the case of training binary classifiers that do not allow multi-class classification). Once these values have been calculated, they can be combined into a single value that will be used to evaluate the performance of the classifier. In this regard, there are 3 strategies: macro, weighted, and micro. We will use only the first two:

- **Macro**. In this strategy, those metrics such as the PPV or the F-score are calculated as the arithmetic mean of the metrics of each class. As it is an arithmetic average, it does not consider the possible imbalance between classes.
- **Weighted**. In this stratey, the metrics corresponding to each class are averaged, weighting them with the number of patterns that belong (desired output) to each class. It is therefore suitable when classes are unbalanced.
- **Micro**. TP, FN, and FP are calculated globally. When the classes are not mutually exclusive, the micro-PPV or micro-F-score is equal to the accuracy value. Therefore, this metric is useful when there are mutually exclusive classes. 

In this assignment, you are asked to:

1. Develop a function called `confusionMatrix` (same name as in the previous assignment) that returns the values of the metrics adapted to the condition of having more than two classes. To do so, include an additional parameter that allows to calculate them in the *macro* and *weighted* forms.

    This function should receive two matrices: model outputs (`outputs`) and desired outputs (`targets`), both of Boolean elements and dimension 2, with each pattern in a row and each class in a column. The first thing this function should do is to check that the number of columns of both matrices is equal and is different from 2. In case they have only one column, these columns are taken as vectors and the confusionMatrix function developed in the previous assignment is called.
    
    ### Question 4.12
    
    > ❓ Why are two-column matrices invalid?
    
    `Two-column matrices are invalid because the original confusionMatrix function for binary classification already handles the case of two classes. If the input has exactly two columns, it would be ambiguous whether to treat it as a binary problem (call the old function) or as a multi-class problem. To avoid this conflict, the new function requires either more than two columns (multi-class) or one column (binary, treated as a vector), ensuring a clear distinction between binary and multi-class cases.`
    
    If both matrices have more than 2 columns, the following steps can be followed:
    
    - Reserve memory for the sensitivity, specificity, PPV, NPV and F-score vectors, with one value per class, initially equal to 0. To do this, the `zeros` function can be used.
    
    - Iterate for each class, and, if there are patterns in that class, make a call to the `confusionMatrix` function of the previous assignment, passing as vectors the columns corresponding to the class of that iteration of the outputs and targets matrices. Assign the result to the corresponding element of the sensitivity, specificity, PPV, NPV and $F_1$ vectors.
    - Reserve memory for the confusion matrix.
    - Perform a nested loop where both loops iterate over the class indices to fill in each cell of the confusion matrix.
  ### Question 4.13 
  > 💡 In reality, it is not necessary to reserve memory and use nested loops.  
  > This can be implemented in a more elegant and efficient way using a **comprehension**, in a single line of code.  
  > As a hint, it may help to first write the nested loop with a one-line body, and then try to convert it into a comprehension.

 
* Aggregate the values of sensitivity, specificity, PPV, NPV, and $F_1$-score for each class into a single value according to the *macro* or *weighed* strategy, as specified in the input argument.
> ⚠️ When using the **weighted** strategy, it is necessary to compute how many instances belong to each class in order to calculate the weighted average.  
> This can be done with `sum(targets, dims=1)`.  
> However, this call returns a matrix with a single row (and one column per class).  
> If you try to compute the element-wise product of this result with one of the metric vectors (e.g., the sensitivity vector), the result will be a matrix, because the metric vector is interpreted as a column — and multiplying a row by a column gives a full matrix.
> ✅ This can be resolved in several ways, the simplest being to **flatten** the matrix of instance counts into a vector using: `vec(sum(targets, dims=1))`
    
* Finally, calculate the accuracy value with the `accuracy` function developed in a previous assignment, and calculate the error rate from this value.

In [21]:
# HELPER

using Statistics

# function from previous assignment
function confusionMatrix(outputs::AbstractArray{Bool,1}, targets::AbstractArray{Bool,1})
    @assert length(outputs) == length(targets) "Outputs and targets must have the same length"
    
    # True positives, true negatives, false positives, false negatives
    TP = sum(outputs .& targets)
    TN = sum(.!outputs .& .!targets)
    FP = sum(outputs .& .!targets)
    FN = sum(.!outputs .& targets)

    # Confusion matrix: rows = predicted, columns = actual
    cm = [TP FP; FN TN]

    # Accuracy and error rate
    accuracy = (TP + TN) / length(targets)
    error_rate = 1 - accuracy

    # Sensitivity (recall)
    sensitivity = if TP + FN == 0
        # All targets are negative
        1.0
    else
        TP / (TP + FN)
    end

    # Specificity
    specificity = if TN + FP == 0
        # All targets are positive
        1.0
    else
        TN / (TN + FP)
    end

    # Positive predictive value (precision)
    ppv = if TP + FP == 0
        if TP + FN == 0  # all patterns negative
            1.0
        else
            0.0
        end
    else
        TP / (TP + FP)
    end

    # Negative predictive value
    npv = if TN + FN == 0
        if TN + FP == 0  # all patterns positive
            1.0
        else
            0.0
        end
    else
        TN / (TN + FN)
    end

    # F-score
    fscore = if sensitivity + ppv == 0
        0.0
    else
        2 * (ppv * sensitivity) / (ppv + sensitivity)
    end

    return (
        accuracy,
        error_rate,
        sensitivity,
        specificity,
        ppv,
        npv,
        fscore,
        cm
    )
end

confusionMatrix (generic function with 5 methods)

In [22]:

function confusionMatrix(outputs::AbstractArray{Bool,2}, targets::AbstractArray{Bool,2}; weighted::Bool=true)
    #TODO
    
    # --- 1. Sanity checks ---
    n_classes_output = size(outputs, 2)
    n_classes_target = size(targets, 2)

    @assert n_classes_output == n_classes_target "Outputs and targets must have the same number of columns (classes)."

    # The first thing this function should do is to check that the number of columns of both matrices is equal and is different 
    # from 2. In case they have only one column,these columns are taken as vectors and the confusionMatrix function 
    # developed in the previous assignment is called.

    n_classes_output = size(outputs, 2)
    n_classes_target = size(targets, 2)
    @assert n_classes_output == n_classes_target "Outputs and targets must have the same number of columns (classes)."
    @assert n_classes_target >= 1 "Outputs and targets must have at least one column (class)."

    # --- 2. If single-column (binary case), call previous 1D confusionMatrix ---
    if n_classes_output == 1
        # Take columns as vectors
        y_pred = vec(outputs)
        y_true = vec(targets)
        return confusionMatrix(y_pred, y_true)  # call the binary version
    end
    
    # --- 3. Multi-class/multi-label
    n_classes = size(targets, 2)
    print("n_classes , $n_classes \n")
    # Initialize vectors to store metrics per class
    sensitivities = zeros(Float64, n_classes)
    specificities = zeros(Float64, n_classes)
    positive_predicted_values = zeros(Float64, n_classes)
    negative_predicted_values = zeros(Float64, n_classes)
    f1_scores = zeros(Float64, n_classes)

    for class in 1:n_classes
        y_c_pred = outputs[:, class]
        y_c_true = targets[:, class]

        #Counts how many elements are true in both y_c_pred and y_c_true
        true_pos = sum(y_c_pred .& y_c_true) # "Apply the operation element by element across the two arrays."
                                             # & is the element-wise AND operator in Julia.

        # Counts how many samples are not predicted as class c and are not actually of class c
        true_neg = sum(.!y_c_pred .& .!y_c_true)

        # Counts how many samples are predicted as class c but are not actually of class c
        false_pos = sum(y_c_pred .& .!y_c_true)

        # Counts how many samples are not predicted as class c but are actually of class c
        false_neg = sum(.!y_c_pred .& y_c_true)

        print("class : $class y_c_true : $y_c_true\n")
        print("class : $class y_c_pred : $y_c_pred\n")
        print("class : $class true_pos : $true_pos\n")
        print("class : $class true_neg : $true_neg\n")
        print("class : $class false_pos : $false_pos\n")
        print("class : $class false_neg : $false_neg\n")
        print("\n")    
        
        
        sensitivities[class] = if true_pos + false_neg == 0
            # All targets are negative
            1.0
        else
            true_pos / (true_pos + false_neg)
        end

        specificities[class] = if true_neg + false_pos == 0
            # All targets are positive
            1.0
        else
            true_neg / (true_neg + false_pos)
        end
        
        #Precision = True Positives / (True Positives + False Positives
        positive_predicted_values[class] = if true_pos + false_pos == 0
            if true_pos + false_neg == 0
                # All targets are negative
                1.0
            else
                # No positive predictions, but there are positive targets
                0.0
            end
        else
            true_pos / (true_pos + false_pos)
        end


        #Negative Predictive Value = True Negatives / (True Negatives + False Negatives)
        negative_predicted_values[class] = if true_neg + false_neg == 0
            if true_neg + false_pos == 0
                # All targets are positive
                1.0
            else
                # No negative predictions, but there are negative targets
                0.0
            end
        else
            true_neg / (true_neg + false_neg)
        end

        #F1 Score = 2 * (Precision * Sensitivity) / (Precision + Sensitivity)
        f1_scores[class] = if sensitivities[class] + positive_predicted_values[class] == 0
            0.0
        else
            2 * (positive_predicted_values[class] * sensitivities[class]) / (positive_predicted_values[class] + sensitivities[class])
        end
    end 
    print("sensitivities : $sensitivities\n")
    print("specificities : $specificities\n")
    print("positive_predicted_values : $positive_predicted_values\n")
    print("negative_predicted_values : $negative_predicted_values\n")
    print("f1_scores : $f1_scores\n")

    
    # Overall accuracy and error rate
    #accuracy = mean(outputs .== targets)
    # In this case (multi-class), we should count whether the whole row was predicted correctly, not each value.
    accuracy = mean([argmax(outputs[i, :]) == argmax(targets[i, :]) for i in 1:size(targets, 1)])
    error_rate = 1 - accuracy
    print("accuracy : $accuracy\n")
    print("error_rate : $error_rate\n")

    # Aggregate metrics
-   # **Weighted**. In this stratey, the metrics corresponding to each class are averaged, weighting them with 
    # the number of patterns that belong (desired output) to each class. It is therefore suitable when classes are unbalanced.

    if weighted

        # When using the **weighted** strategy, it is necessary to compute how many instances belong 
        # to each class in order to calculate the weighted average.  
        class_counts = vec(sum(targets, dims=1))  # flatten row matrix into vector
        total_instances = sum(class_counts)
        weights = class_counts / total_instances
        sensitivity = sum(weights .* sensitivities)
        specificity = sum(weights .* specificities)
        positive_predicted_values = sum(weights .* positive_predicted_values)
        negative_predicted_values = sum(weights .* negative_predicted_values)
        fscore = sum(weights .* f1_scores)

    # - **Macro**. In this strategy, those metrics such as the PPV or the F-score are calculated as the arithmetic mean of the metrics of each class.
    #  As it is an arithmetic average, it does not consider the possible imbalance between classes.
    else
        sensitivity = mean(sensitivities)
        specificity = mean(specificities)
        positive_predicted_values = mean(positive_predicted_values)
        negative_predicted_values = mean(negative_predicted_values)
        fscore = mean(f1_scores)    
    end

    cm = [sum(targets[:, i] .& outputs[:, j]) for i in 1:n_classes, j in 1:n_classes]


    return (
        accuracy,
        error_rate,
        sensitivity,
        specificity,
        positive_predicted_values,
        negative_predicted_values,
        fscore,
        cm
    )
    
end


# Example usage
outputs = Bool[
    1 0 0;
    0 1 0;
    0 0 1;
    1 0 0
]

targets = Bool[
    1 0 0;
    0 1 0;
    0 1 0;
    1 0 0
]
metrics= confusionMatrix(outputs, targets, weighted=true)
display(metrics)

n_classes , 3 
class : 1 y_c_true : Bool[1, 0, 0, 1]
class : 1 y_c_pred : Bool[1, 0, 0, 1]
class : 1 true_pos : 2
class : 1 true_neg : 2
class : 1 false_pos : 0
class : 1 false_neg : 0

class : 2 y_c_true : Bool[0, 1, 1, 0]
class : 2 y_c_pred : Bool[0, 1, 0, 0]
class : 2 true_pos : 1
class : 2 true_neg : 2
class : 2 false_pos : 0
class : 2 false_neg : 1

class : 3 y_c_true : Bool[0, 0, 0, 0]
class : 3 y_c_pred : Bool[0, 0, 1, 0]
class : 3 true_pos : 0
class : 3 true_neg : 3
class : 3 false_pos : 1
class : 3 false_neg : 0

sensitivities : [1.0, 0.5, 1.0]
specificities : [1.0, 1.0, 0.75]
positive_predicted_values : [1.0, 1.0, 0.0]
negative_predicted_values : [1.0, 0.6666666666666666, 1.0]
f1_scores : [1.0, 0.6666666666666666, 0.0]
accuracy : 0.75
error_rate : 0.25


(0.75, 0.25, 0.75, 1.0, 1.0, 0.8333333333333333, 0.8333333333333333, [2 0 0; 0 1 1; 0 0 0])

This function should return a tuple with the same values specified in the previous exercise, in the same order: **accuracy, error rate, sensitivity, specificity, PPV, NPV, F1-score, and the confusion matrix.** As mentioned before, the use of loops is allowed for implementing this function.


3. Develop another function called `confusionMatrix` in which the first parameter `outputs` is of type `AbstractArray{<:Real,2}`, and `targets` is of type `AbstractArray{Bool,2}` (the same as before). What this function should do is to convert the first parameter to an array of boolean values (using the function `classifyOutputs`) and call the previous function. Additionally, this function may receive two optional arguments: `threshold` and `weighted`.  
These values will be used when calling the respective underlying functions that this one builds upon.

  ### Question 4.14
> ❓ Within this function, in which calls will each of these two optional parameters be used?

`The threshold parameter is used when calling the classifyOutputs function, since it determines how continuous outputs are converted into boolean predictions (e.g., classifying probabilities as true or false).
The weighted parameter is used when calling the boolean version of the confusionMatrix function, to indicate whether the overall metrics should be computed as a weighted average or as a simple average across classes.`

> ❓ In which cases is the `threshold` parameter required?

`The threshold parameter is required when the model outputs real numbers instead of boolean values.
It defines the cutoff above which outputs are classified as true (positive) and below which as false (negative).
If the outputs are already boolean or if it’s a multi-class problem, the threshold parameter is not used.`

Since this function builds upon the previous one, it **should not contain any loops**.

In [23]:
# HELPER

function classifyOutputs(outputs::AbstractArray{<:Real,2}; 
                        threshold::Real=0.5) 
   numOutputs = size(outputs, 2);
    @assert(numOutputs!=2)
    if numOutputs==1
        return outputs.>=threshold;
    else
        # Look for the maximum value using the findmax funtion
        (_,indicesMaxEachInstance) = findmax(outputs, dims=2);
        # Set up then boolean matrix to everything false while max values aretrue.
        outputs = falses(size(outputs));
        outputs[indicesMaxEachInstance] .= true;
        # Defensive check if all patterns are in a single class
        @assert(all(sum(outputs, dims=2).==1));
        return outputs;
    end;
end;


In [24]:
function confusionMatrix(outputs::AbstractArray{<:Real,2},targets::AbstractArray{Bool,2}; threshold::Real=0.5, weighted::Bool=true)
    #TODO

    # Convert continuous outputs into Boolean predictions
    classified_outputs = classifyOutputs(outputs, threshold=threshold)

    # Call the Boolean version of confusionMatrix
    return confusionMatrix(classified_outputs, targets; weighted=weighted)

end


confusionMatrix (generic function with 5 methods)

In [25]:
outputs_real = [
    0.9 0.1 0.0;
    0.2 0.8 0.1;
    0.1 0.2 0.7;
    0.95 0.05 0.02
]

targets = Bool[
    1 0 0;
    0 1 0;
    0 1 0;
    1 0 0
]

results = confusionMatrix(outputs_real, targets; threshold=0.5, weighted=true)
display(results)


n_classes , 3 
class : 1 y_c_true : Bool[1, 0, 0, 1]
class : 1 y_c_pred : Bool[1, 0, 0, 1]
class : 1 true_pos : 2
class : 1 true_neg : 2
class : 1 false_pos : 0
class : 1 false_neg : 0

class : 2 y_c_true : Bool[0, 1, 1, 0]
class : 2 y_c_pred : Bool[0, 1, 0, 0]
class : 2 true_pos : 1
class : 2 true_neg : 2
class : 2 false_pos : 0
class : 2 false_neg : 1

class : 3 y_c_true : Bool[0, 0, 0, 0]
class : 3 y_c_pred : Bool[0, 0, 1, 0]
class : 3 true_pos : 0
class : 3 true_neg : 3
class : 3 false_pos : 1
class : 3 false_neg : 0

sensitivities : [1.0, 0.5, 1.0]
specificities : [1.0, 1.0, 0.75]
positive_predicted_values : [1.0, 1.0, 0.0]
negative_predicted_values : [1.0, 0.6666666666666666, 1.0]
f1_scores : [1.0, 0.6666666666666666, 0.0]
accuracy : 0.75
error_rate : 0.25


(0.75, 0.25, 0.75, 1.0, 1.0, 0.8333333333333333, 0.8333333333333333, [2 0 0; 0 1 1; 0 0 0])

4. Override this function once again by developing another function of the same name that performs the same task, but this time taking as inputs two vectors (`targets` and `outputs`) of the same length, whose elements are of any type (i.e., they are of type `AbstractArray{<:Any}`), plus the additional parameter that allows to aggregate the metrics through the *macro* and *weighted* strategies. The vectors `outputs` and `targets` contain the predicted and true labels for each instance, respectively. Therefore, both vectors must have the same length. The vector `classes` contains all possible class labels, and will usually have a different length than `outputs` and `targets`. The elements of these vectors represent labels and may take various forms — such as real numbers, integers, strings, or symbols. For this reason, the type `Any` is used. For example, a valid `classes` vector could be:
   ```julia
       ["dog", "cat", 3, :green]
    ```
    Obviously, both the predicted labels (`outputs` vector) and the true labels (`targets` vector) must be included in the `classes` vector.

  ### Question 4.15
 > ❓ Write this check without using any loops. You may find the functions `all`, `in`, and `unique` useful. (A solution is provided at the end of the notebook.)

In [26]:
classes = ["dog", "cat", 3, :green]
targets = ["dog", "cat", "cat", "dog"]
outputs = ["cat", "cat", "dog", "dog"]

@assert (all([in(output, classes) for output in unique(outputs)]))
@assert (all([in(target, classes) for target in unique(targets)]))
# The solution provided at the end does not check if the outputs and targets are included in the classs vector

This function can be implemented by calling a previous version of `confusionMatrix`, specifically the one that receives **boolean matrices (2D)** along with an optional argument.

To prepare for this, both `outputs` and `targets` should be **one-hot encoded** by calling the `oneHotEncoding` function, passing the `classes` vector as input.

Once both matrices are encoded, you can call the corresponding `confusionMatrix` function.
⚠️ It is essential to build the `classes` vector **before** the calls to `oneHotEncoding`, and to use the **same vector** in both calls.  

   ### Question 4.16
  > ❓ What could happen if this is not done?

`If the same classes vector is not used for both outputs and targets when calling oneHotEncoding, the resulting one-hot matrices could have different numbers of columns or mismatched class orders.
That means each column might represent a different class in outputs and targets, so comparisons between them (for true/false positives, etc.) would be meaningless.
As a result, the computed confusion matrix and all derived metrics (accuracy, sensitivity ...) would be incorrect or completely inconsistent.`

In [27]:
# HELPER
function oneHotEncoding(feature::AbstractArray{<:Any,1},      
        classes::AbstractArray{<:Any,1})

    """
    Parameters
    ----------
    feature :: AbstractVector
        The input vector of categorical values to be encoded.
    classes :: AbstractVector
        The list/array of unique classes used as encoding reference.
    """
    """
    AbstractArray{<:Any,1} :
    AbstractArray → not restricted to just Vector, could be any 1-dimensional array type.
    1 → one-dimensional array (i.e. a vector).
    <:Any → element type can be any subtype of Any (which basically means no restriction at all).
    So effectively, AbstractArray{<:Any,1} is just a very general way of saying:
    👉 “Accept any 1-D array, regardless of element type.”
    """
    # Defensive: ensure feature is a vector
    # Check that all feature values exist in the set of classes
    @assert(all([in(value, classes) for value in feature]))
    
    # Number of classes
    numClasses = length(classes)
    
    # Defensive: require at least two classes
    @assert(numClasses > 1)
    
    if (numClasses == 2)
        # Special case: binary classification, use a single column
        oneHot = reshape(feature .== classes[1], :, 1)
    else
        # General case: more than two classes
        oneHot = BitArray{2}(undef, length(feature), numClasses)
        for numClass = 1:numClasses
            # Mark 1 where feature matches the class
            oneHot[:, numClass] .= (feature .== classes[numClass])
        end
    end


    """
    Returns
    -------
    AbstractArray
        A one-hot encoded array:
        - Shape (n, 1) if there are 2 classes (binary case).
        - Shape (n, numClasses) if there are more than 2 classes.
    """
    return oneHot
end

oneHotEncoding (generic function with 1 method)

In [28]:
function confusionMatrix(outputs::AbstractArray{<:Any,1}, targets::AbstractArray{<:Any,1}, classes::AbstractArray{<:Any,1}; weighted::Bool=true)
    # --- 1. Sanity checks ---
    @assert length(outputs) == length(targets) "Outputs and targets must have the same length."
    @assert (all([in(output, classes) for output in unique(outputs)])) "All output labels must exist in classes."
    @assert (all([in(target, classes) for target in unique(targets)])) "All target labels must exist in classes."

    # --- 2. One-hot encode both using the same classes vector ---
    outputs_onehot = oneHotEncoding(outputs, classes)
    targets_onehot = oneHotEncoding(targets, classes)

    # --- 3. Call the previous confusionMatrix version (the 2D boolean one) ---
    return confusionMatrix(outputs_onehot, targets_onehot; weighted=weighted)

end

confusionMatrix (generic function with 5 methods)

In [29]:
classes = ["dog", "cat", 3, :green]
targets = ["dog", "cat", "cat", "dog"]
outputs = ["cat", "cat", "dog", "dog"]

metrics = confusionMatrix(outputs, targets, classes; weighted=true)
display(metrics)


n_classes , 4 
class : 1 y_c_true : Bool[1, 0, 0, 1]
class : 1 y_c_pred : Bool[0, 0, 1, 1]
class : 1 true_pos : 1
class : 1 true_neg : 1
class : 1 false_pos : 1
class : 1 false_neg : 1

class : 2 y_c_true : Bool[0, 1, 1, 0]
class : 2 y_c_pred : Bool[1, 1, 0, 0]
class : 2 true_pos : 1
class : 2 true_neg : 1
class : 2 false_pos : 1
class : 2 false_neg : 1

class : 3 y_c_true : Bool[0, 0, 0, 0]
class : 3 y_c_pred : Bool[0, 0, 0, 0]
class : 3 true_pos : 0
class : 3 true_neg : 4
class : 3 false_pos : 0
class : 3 false_neg : 0

class : 4 y_c_true : Bool[0, 0, 0, 0]
class : 4 y_c_pred : Bool[0, 0, 0, 0]
class : 4 true_pos : 0
class : 4 true_neg : 4
class : 4 false_pos : 0
class : 4 false_neg : 0

sensitivities : [0.5, 0.5, 1.0, 1.0]
specificities : [0.5, 0.5, 1.0, 1.0]
positive_predicted_values : [0.5, 0.5, 1.0, 1.0]
negative_predicted_values : [0.5, 0.5, 1.0, 1.0]
f1_scores : [0.5, 0.5, 1.0, 1.0]
accuracy : 0.5
error_rate : 0.5


(0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, [1 1 0 0; 1 1 0 0; 0 0 0 0; 0 0 0 0])

Since this function builds on a previous one, it must **return the same outputs** and **must not use loops**.

5. You must now create a final version of `confusionMatrix`, similar to the one described above, but **without receiving the `classes` vector** explicitly. Instead, `classes` should be constructed from `outputs` and `targets` using:
   ```julia
          classes = unique(vcat(targets, outputs))
   ```
  Then, call the previous version of `confusionMatrix`, passing `outputs`, `targets`, `classes`, and the optional argument.

> ⚠️ Use this version with caution.
> If the data has been partitioned (e.g., for cross-validation), some classes might not appear in `outputs` or `targets`, potentially affecting results.

In [30]:
function confusionMatrix(outputs::AbstractArray{<:Any,1}, targets::AbstractArray{<:Any,1}; weighted::Bool=true)
    #TODO
    # --- 1. Construct the classes vector from both targets and outputs ---
    classes = unique(vcat(targets, outputs))

    # --- 2. Call the previous version (that expects classes explicitly) ---
    return confusionMatrix(outputs, targets, classes; weighted=weighted)

    
end

confusionMatrix (generic function with 5 methods)

In [31]:
classes = ["dog", "cat", 3, :green]
targets = ["dog", "cat", "cat", "dog"]
outputs = ["cat", "cat", "dog", "dog"]

metrics = confusionMatrix(outputs, targets; weighted=true)
display(metrics)


(0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, [1 1; 1 1])

As before, this function must return the same outputs and should not use loops.

## printConfusionMatrix (multiclass versions)
As in the previous notebook, implement four versions of the `printConfusionMatrix` function — now for the multiclass case, where both `outputs` and `targets` may be matrices or vectors, depending on the context.

In [32]:
function printConfusionMatrix(outputs::AbstractArray{Bool,2}, targets::AbstractArray{Bool,2}; weighted::Bool=true)
    #TODO
    acc, err, sens, spec, ppv, npv, f1, cm = confusionMatrix(outputs, targets; weighted=weighted)

    println("=== Confusion Matrix Metrics ===")
    println("Accuracy: ", acc)
    println("Error rate: ", err)
    println("Sensitivity (Recall): ", sens)
    println("Specificity: ", spec)
    println("Positive Predictive Value (Precision): ", ppv)
    println("Negative Predictive Value: ", npv)
    println("F-score: ", f1)
    println("Confusion Matrix:\n", cm)

end

printConfusionMatrix (generic function with 3 methods)

In [33]:
function printConfusionMatrix(outputs::AbstractArray{<:Real,2}, targets::AbstractArray{Bool,2}; weighted::Bool=true)
    #TODO
    acc, err, sens, spec, ppv, npv, f1, cm = confusionMatrix(outputs, targets; weighted=weighted)

    println("=== Confusion Matrix Metrics ===")
    println("Accuracy: ", acc)
    println("Error rate: ", err)
    println("Sensitivity (Recall): ", sens)
    println("Specificity: ", spec)
    println("Positive Predictive Value (Precision): ", ppv)
    println("Negative Predictive Value: ", npv)
    println("F-score: ", f1)
    println("Confusion Matrix:\n", cm)
end

printConfusionMatrix (generic function with 3 methods)

In [34]:
function printConfusionMatrix(outputs::AbstractArray{<:Any,1}, targets::AbstractArray{<:Any,1}, classes::AbstractArray{<:Any,1}; weighted::Bool=true)
    #TODO
    acc, err, sens, spec, ppv, npv, f1, cm = confusionMatrix(outputs, targets, classes; weighted=weighted)

    println("=== Confusion Matrix Metrics ===")
    println("Accuracy: ", acc)
    println("Error rate: ", err)
    println("Sensitivity (Recall): ", sens)
    println("Specificity: ", spec)
    println("Positive Predictive Value (Precision): ", ppv)
    println("Negative Predictive Value: ", npv)
    println("F-score: ", f1)
    println("Confusion Matrix:\n", cm)
end

printConfusionMatrix (generic function with 3 methods)

In [35]:
function printConfusionMatrix(outputs::AbstractArray{<:Any,1}, targets::AbstractArray{<:Any,1}; weighted::Bool=true)
    #TODO
    acc, err, sens, spec, ppv, npv, f1, cm = confusionMatrix(outputs, targets; weighted=weighted)

    println("=== Confusion Matrix Metrics ===")
    println("Accuracy: ", acc)
    println("Error rate: ", err)
    println("Sensitivity (Recall): ", sens)
    println("Specificity: ", spec)
    println("Positive Predictive Value (Precision): ", ppv)
    println("Negative Predictive Value: ", npv)
    println("F-score: ", f1)
    println("Confusion Matrix:\n", cm)
end

printConfusionMatrix (generic function with 4 methods)

All four versions must also accept an optional argument to choose whether metrics are computed using the macro or weighted strategy.

> 🛠 These four functions will not be graded, but they will be useful for Exercise 2.

### Learn Julia

The defensive programming line to ensure that all classes of the `output` vector are included in the desired output vector is as follows:

```Julia
@assert(all([in(output, unique(targets)) for output in outputs]))
```